# Trading Correlation with Causation

<a href="https://www.kaggle.com/code/addarm/trading-correlation-with-causation" target="_blank" rel="noopener noreferrer">
  <img src="https://www.kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle" title="Open in Kaggle">
</a>

![OpenAI Causal Graphs](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/causality_graphs_finance.png)

Correlation is not causation. We hear that often, but what does it mean when money is at stake?

Imagine sitting on a risk committee deciding whether to raise a fund's leverage. Leverage means investment exposure relative to the fund's own capital: a fund holding $3 of assets for each dollar of capital is more leveraged than one holding $1.50. We examine records for 1,000 funds, half running high leverage and half running low leverage.

Over the next quarter, only 28% of the high-leverage funds suffered a large loss, compared with 42% of the low-leverage funds. Should the committee recommend raising leverage? The numbers make it tempting—but what, exactly, are they comparing?

By the end, you should be able to distinguish an observed relationship from the effect of changing a policy, choose controls using a causal graph, and explain the assumptions needed to connect the two. All fund data here are synthetic teaching examples, not evidence about actual funds.

## A reversal inside the data

Before the committee votes, we learn one more thing.

Each fund had a funding and liquidity position, measured *before* the leverage decision. A healthy fund has stable funding and can meet margin calls without selling assets. An impaired fund is more vulnerable to funding stress. Healthy funds were more likely to choose high leverage; impaired funds mostly chose low leverage.

We will use three letters throughout: $C=1$ means healthy funding and $C=0$ impaired funding; $D=1$ means high leverage and $D=0$ low leverage; $Y=1$ means a large loss during the following quarter.

Here is the constructed record, separated by funding health. Each fraction is the number of funds that suffered a large loss divided by the number of funds in that group.



| Funding health | High leverage $D=1$ | Low leverage $D=0$ |
|---|---:|---:|
| Healthy, $C=1$ | 80 / 400 = 20% | 10 / 100 = 10% |
| Impaired, $C=0$ | 60 / 100 = 60% | 200 / 400 = 50% |
| Pooled | 140 / 500 = **28%** | 210 / 500 = **42%** |

### A few terms to keep handy

We have found a puzzle: high leverage appears safer overall, yet the opposite holds within each funding-health group. These terms will help us investigate it. There is no need to memorize them—we will introduce each properly when we need it.

| Term | Meaning in this article |
|---|---|
| **Association** | A pattern in observed data; it does not by itself establish cause. |
| **Confounder** | A common cause of a decision and its outcome. Here, funding health affects both leverage choices and losses. |
| **Causal graph (DAG)** | Arrows express hypothesized causal directions, without directed cycles. |
| **Backdoor path** | A route from decision to outcome beginning with an arrow *into* the decision, such as leverage $\leftarrow$ funding health $\rightarrow$ losses. |
| **Conditioning** | Comparing funds within the same group, such as comparing only healthy funds. It is not always a safe adjustment. |
| **Intervention, $do(D=d)$** | Setting a policy instead of observing which policy a fund chose. |
| **Collider** | A common effect of two variables. Selecting funds because of that common effect can create a misleading association. |
| **Identification** | Deriving the effect of an intervention from observable distributions, subject to explicit assumptions. |

The question to keep in mind: **what happened among funds that chose high leverage, and what would happen if we required them to choose it?**

### Run once: notebook setup

The next cell imports the libraries needed for the experiments. The figures are included as Markdown images so the notebook stays focused on causal reasoning, not plotting code.

In [ ]:
from fractions import Fraction as F
from itertools import product

import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm

SEED = 42
EDGES = [("C", "D"), ("C", "Y"), ("D", "Y")]


In [ ]:
cells = pd.DataFrame({"C": [1, 1, 0, 0], "D": [1, 0, 1, 0], "losses": [80, 10, 60, 200], "funds": [400, 100, 100, 400]})
rows = []
for r in cells.itertuples():
    rows += [(r.C, r.D, 1)] * r.losses + [(r.C, r.D, 0)] * (r.funds - r.losses)
funds = pd.DataFrame(rows, columns=["C", "D", "Y"])
by_cell = funds.groupby(["C", "D"])["Y"].agg(losses="sum", funds="size", rate="mean")
pooled = funds.groupby("D")["Y"].agg(losses="sum", funds="size", rate="mean")
print(by_cell.to_string(formatters={"rate": "{:.0%}".format}))
print(pooled.to_string(formatters={"rate": "{:.0%}".format}))
naive = pooled.loc[1, "rate"] - pooled.loc[0, "rate"]
print(f"Pooled high minus low: {naive:+.2f}")
assert len(funds) == 1000 and abs(naive - (-0.14)) < 1e-12

![Reversal Plot](images/causality_assets/fig1_simpson_reversal.png)

High leverage loses more inside both groups (20% against 10% among healthy funds, 60% against 50% among impaired ones) and less in the pool. 

The pool is misleading because of composition: 400 of the 500 high-leverage funds were healthy (80%), against 100 of the 500 low-leverage funds (20%), and healthy funds lose less whatever they do.

This is a **Simpson reversal**. The pooled number describes how the funds that chose high leverage fared. It doesn't say what leverage did to them. So the question to chase is who chose high leverage, and why.

## Drawing the story

Funding health separated the groups, and timing and economics tell us how. 

Health $C$ is fixed before the policy decision, so it could influence the decision but not the other way round. A healthy fund can carry extra exposure without stress, so $C\to D$. It's also less likely to be forced into fire sales, so $C\to Y$. And leverage itself might change the chance of a large loss, $D\to Y$, the arrow the committee cares about.

We draw the story as a directed acyclic graph, or DAG, where each node is a variable, each arrow points from a cause to its effect, and following arrows never leads back to where you started. A path is any sequence of connected arrows, followed in either direction. Because $C$ causes both the policy and the outcome, it's a confounder. Which path could make high leverage look safe even if leverage did nothing?

![Funds DAG](images/causality_assets/fig2_funds_dag.png)

In Figure 2 we see that funding health affects both the leverage choice and the chance of a large loss; the graph expresses assumptions, not established empirical facts.

Healthy funds both choose high leverage and lose less, so leverage and low losses travel together along $D\leftarrow C\rightarrow Y$ whatever $D\to Y$ does. That's confounding. The arrows are claims we make: the table can't confirm any direction, so we take them from timing and economics, and the graph is where our assumptions sit and where others can dispute them.

## Observations versus Intervention

The first operation is observation. Pick the funds with $D=1$ and count their large losses. That's conditioning on $D=1$, and it gives $P(Y=1\mid D=1)$, the probability of a large loss among those funds, or 28% in our synthetic data.

The second operation is intervention. Require every fund, healthy or impaired, to run high leverage, then count large losses. We write that probability as $P(Y=1\mid do(D=1))$, where $do(D=1)$ means $D$ is set to 1 from outside, overruling whatever would have chosen it. The funds and their natural mix stay as they were, and only the policy changes.

In the graph, setting $D$ deletes the arrow into $D$, because the policy no longer listens to $C$. Every other arrow stays, as the right panel below shows.

![Observation versus intervention](images/causality_assets/fig3_observation_vs_intervention.png)

In Figure 3, observing high leverage selects a type of fund; setting high leverage removes its natural assignment mechanism.

For each fund, let $Y(1)$ be the large-loss indicator it **would** have under high leverage and $Y(0)$ the one it would have under low leverage. In reality, we observe only the outcome under the policy it followed. The committee's target is the **average treatment effect (ATE)**, $\tau=E[Y(1)-Y(0)]$. Since $Y$ is a loss indicator, $\tau=0.10$ means high leverage raises the population's large-loss probability by 10 percentage points.

This simple notation assumes **no interference**: changing one fund's policy does not change another fund's losses. In real markets that can fail, because funds trade overlapping assets and forced sales can move prices. Our toy model holds those spillovers aside. A market-wide policy change would require an additional model of feedback and price effects.

To examine the intervention rather than assume its result, we need a model in which we can actually set the policy.

## A model we can intervene on

A **structural causal model (SCM)** writes each variable as a function of its direct causes plus randomness from outside the model. We build ours one assignment at a time, in the order of the graph - remember all numbers come from the fictional table above.

Funding health $C$ has no causes in the graph. Let $U_C$ be a uniform random number between 0 and 1, an exogenous variable that the model takes as given. The indicator $\mathbf 1\{\cdot\}$ equals 1 when its condition holds and 0 otherwise, so calling a fund healthy when $U_C<0.5$ makes half the funds healthy.

The policy $D$ listens to $C$ and to its own randomness $U_D$. A healthy fund chooses high leverage with probability 0.8, an impaired fund with probability 0.2.

The outcome $Y$ listens to both, plus $U_Y$. An impaired, low-leverage fund has a 0.5 chance of a large loss, high leverage adds 0.1, and health subtracts 0.4. These reproduce the table's four cell rates.

Written out in order, the three assignments are:

$$
\begin{aligned}
C&=\mathbf 1\{U_C<0.5\}\\
D&=\mathbf 1\{U_D<0.2+0.6\,C\}\\
Y&=\mathbf 1\{U_Y<0.5+0.1\,D-0.4\,C\}
\end{aligned}
$$

We assume $U_C,U_D,U_Y$ are mutually independent. An intervention changes one line. Under $do(D=d)$ we delete the equation for $D$ and write $D:=d$, leaving the other equations and all the randomness untouched. Filtering to $D=d$ does something else: it keeps the equation for $D$ and picks the funds that happened to satisfy it, which over-represents healthy funds when $d=1$.

Before we simulate a population, we run one fund by hand.

In [ ]:
def fund(u_c, u_d, u_y, p_d_given_c1=0.8, p_d_given_c0=0.2, do=None):
    c = (np.asarray(u_c) < 0.5).astype(int)
    p_d = np.where(c == 1, p_d_given_c1, p_d_given_c0)
    d = (np.asarray(u_d) < p_d).astype(int) if do is None else np.full_like(c, do)
    y = (np.asarray(u_y) < 0.5 + 0.1 * d - 0.4 * c).astype(int)
    return c, d, y

u = (0.30, 0.90, 0.15)
c, d, y = (int(v) for v in fund(*u))
print(f"Noise (U_C, U_D, U_Y) = {u}")
print(f"Observed: C={c}, D={d}, Y={y}   (U_C < 0.5 so healthy; U_D = 0.90 is above 0.8 so low leverage)")
for policy in (1, 0):
    print(f"do(D={policy}): Y={int(fund(*u, do=policy)[2])}   (loss if U_Y < 0.5 + 0.1*{policy} - 0.4*{c} = {0.5 + 0.1 * policy - 0.4 * c:.1f})")

This healthy fund ran low leverage and avoided a loss. On the same noise, setting $D=1$ flips its outcome to a loss and setting $D=0$ reproduces what we saw, so $Y(1)=1$ and $Y(0)=0$ for this fund. We could reuse the noise only because we built the SCM. In real data a fund runs one policy and its other outcome stays hidden, so only group averages can be estimated, under assumptions.

Next, the same model on 200,000 funds, with each simulated value beside the model's exact "SCM value". Predict first: what share of funds is healthy under $do(D=1)$, and what share among the funds observed with $D=1$?

In [ ]:
rng = np.random.default_rng(SEED)
n = 200_000
u_c, u_d, u_y = rng.random((3, n))
c, d_seen, y_seen = fund(u_c, u_d, u_y)
y1, y0 = fund(u_c, u_d, u_y, do=1)[2], fund(u_c, u_d, u_y, do=0)[2]
sim = pd.DataFrame({
    "simulated": [y1.mean(), y0.mean(), (y1 - y0).mean(), y_seen[d_seen == 1].mean(), y_seen[d_seen == 0].mean(), y_seen[d_seen == 1].mean() - y_seen[d_seen == 0].mean(), c.mean(), c[d_seen == 1].mean()],
    "SCM value": [0.40, 0.30, 0.10, 0.28, 0.42, -0.14, 0.5, 0.8],
}, index=["E[Y | do(D=1)]", "E[Y | do(D=0)]", "ATE = E[Y(1)-Y(0)]", "E[Y | D=1]", "E[Y | D=0]", "Observed high minus low", "P(C=1 | do(D=1))", "P(C=1 | D=1)"])
print(sim.round(3).to_string())
assert (sim["simulated"] - sim["SCM value"]).abs().max() < 0.01

Intervening leaves the healthy share at its natural 0.5, since setting a policy can't change pre-decision health. Conditioning on the observed policy gives 0.8. That gap is the **confounding**, and it's why the observed gap is $-14$ pp, while the gap under intervention is $+10$ pp. The simulated values wobble around the exact ones because they're finite-sample averages with monte carlo errors, and the model is true by construction.

## What if we remove the selection?

The $-14$ pp gap came from the selection rule: healthy funds chose high leverage 80% of the time and impaired funds 20%. Call these probabilities $p_{D\mid C=1}$ and $p_{D\mid C=0}$, and keep $P(Y=1\mid C,D)=0.5+0.1D-0.4C$ fixed.

This is a separate constructed population, and the 1,000-fund table is unchanged. We reuse the same draws $U_C,U_D,U_Y$ and change only the selection rule to 0.5 and 0.5, so health no longer influences the policy.

We now predict which of the two effects should change: the observed gap ($-14$ pp) or the effect of setting the policy ($+10$ pp)?

In [ ]:
res = {}
for label, p1, p0 in [("baseline 0.8 / 0.2", 0.8, 0.2), ("balanced 0.5 / 0.5", 0.5, 0.5)]:
    c_, d_, y_ = fund(u_c, u_d, u_y, p1, p0)
    ate = (fund(u_c, u_d, u_y, p1, p0, do=1)[2] - fund(u_c, u_d, u_y, p1, p0, do=0)[2]).mean()
    gap = y_[d_ == 1].mean() - y_[d_ == 0].mean()
    res[label] = (gap, ate)
    print(f"{label}: P(C=1 | D=1) = {c_[d_ == 1].mean():.2f}, observed gap {gap:+.3f}, effect of setting D (ATE) {ate:+.3f}")
(gap_base, ate_base), (gap_flat, ate_flat) = res.values()
assert abs(gap_base + 0.14) < 0.01 and abs(gap_flat - 0.10) < 0.01 and abs(ate_base - 0.10) < 0.01 and abs(ate_flat - 0.10) < 0.01, "Experiment 1 no longer shows -0.14 / +0.10 / ATE 0.10: update the text that quotes them"

The observed gap moved from about $-0.14$ to about $+0.10$. The effect of setting the policy stayed at $+0.10$, identical across the runs because the same $C$ and $U_Y$ feed both. It never depended on who chose each policy. With 0.5 and 0.5, healthy and impaired funds spread equally across the policies, so the observed comparison is a fair one and recovers the effect. Differences in the third decimal are Monte Carlo noise: each rate averages about 100,000 funds, with a standard error near 0.002.

This shows that, inside a model we built, the effect of setting the policy has no dependence on who chose it. It doesn't show that real selection is harmless, or that a real observed gap ever matches an effect without balance like this.

Assume a regulator caps leverage by lottery draw, ignoring fund health. Would the observed gap match the effect of setting the policy? What if the lottery covered only healthy funds?

Yes for the first lottery. It's the 0.5 / 0.5 population: the lottery cuts $C\to D$, the same edit as $do(D=d)$, which is why a randomized experiment needs no adjustment. A healthy-only lottery would match the effect among healthy funds ($+10$ pp here) and say nothing about impaired ones.

Real policies are rarely assigned by lottery, so we have to undo the selection by holding other variables fixed. Some controls help and some do damage. 

How can we tell the difference?

## When does adjusting help—and when does it hurt?

In Figure 2, apart from $D\to Y$, one route links $D$ and $Y$. Which variable would close it if we held it fixed?

It's $D\leftarrow C\rightarrow Y$, closed by $C$. Comparing funds with the same funding health removes the co-movement $D$ and $Y$ get from $C$, as Figure 1 did. A path of this shape is a fork, and a common cause is a good reason to condition. Because it starts with an arrow into $D$, it's also called a backdoor path.

Conditioning isn't always safe. Take $D$ and $Y$ to be independent fair coins. A reporting database admits a fund ($S=1$) when it runs high leverage or suffers a large loss (our own simplification), so $S$ is a collider, a common effect of $D$ and $Y$. Among database funds, what is $P(Y=1\mid D=0,S=1)$? What is it with $D=1$?

In [ ]:
rows = list(product((0, 1), repeat=2))
def p_loss(d, keep=lambda d, y: True):
    sel = [y for dd, y in rows if dd == d and keep(dd, y)]
    return F(sum(sel), len(sel)) if sel else None
database = lambda d, y: d == 1 or y == 1
print(f"All funds:       P(Y=1 | D=0) = {p_loss(0)}   P(Y=1 | D=1) = {p_loss(1)}")
print(f"In the database: P(Y=1 | D=0) = {p_loss(0, database)}   P(Y=1 | D=1) = {p_loss(1, database)}")
assert p_loss(0) == p_loss(1) == F(1, 2) and p_loss(0, database) == 1 and p_loss(1, database) == F(1, 2)

Selection created an association out of nothing. Before selecting, $P(Y=1\mid D)=\tfrac12$ for both policies. In the database every low-leverage fund lost money, because a loss was the only way in, while half the high-leverage funds did. The exact probabilities come from our admission rule, but the general problem appears whenever selection depends on a common effect or one of its descendants.

What if we admit a fund only when it runs high leverage and suffers a large loss. Predict $P(Y=1\mid D=d,S=1)$ for each $d$, then check with `p_loss(d, lambda d, y: d == 1 and y == 1)`.

In this case, only the row $D=1,Y=1$ survives, so $P(Y=1\mid D=1,S=1)=1$, and $P(Y=1\mid D=0,S=1)$ doesn't exist because no low-leverage fund is left. Selecting on a common effect can create an association, and it can delete a cell outright.

We've met a variable to condition on ($C$, a common cause) and one to leave alone ($S$, a common effect). Every path is built from three kinds of junction.

| Junction | Finance example | Open by default? | Conditioning on the middle node |
|---|---|---|---|
| Fork $D\leftarrow C\rightarrow Y$ | Funding health drives leverage and losses | Yes | Blocks it |
| Chain $D\to F\to Y$ | Leverage triggers forced sales $F$, which cause losses | Yes | Blocks it |
| Collider $D\to S\leftarrow Y$ | Leverage and losses both get a fund into the database | No | Opens it (so does conditioning on a descendant of $S$) |

A path is blocked when one of its junctions is closed. Two nodes are d-separated given a set of conditioned variables when every path between them is blocked [1, section 2.4]. In our SCM, d-separation implies conditional independence in the data.

The chain deserves a warning. Conditioning on a mediator like $F$ no longer estimates the total effect. Under further assumptions it may help define a direct effect, but adjustment alone doesn't guarantee that interpretation.

In [ ]:
motifs = {
    "fork": (nx.DiGraph([("C", "D"), ("C", "Y")]), [set(), {"C"}]),
    "chain": (nx.DiGraph([("D", "F"), ("F", "Y")]), [set(), {"F"}]),
    "collider": (nx.DiGraph([("D", "S"), ("Y", "S"), ("S", "S2")]), [set(), {"S"}, {"S2"}]),
}
verdicts = [(name, ", ".join(sorted(z)) or "nothing", nx.is_d_separator(g, {"D"}, {"Y"}, z)) for name, (g, zs) in motifs.items() for z in zs]
print(pd.DataFrame(verdicts, columns=["graph", "condition on", "D and Y d-separated"]).to_string(index=False))
assert [v[2] for v in verdicts] == [False, True, False, True, True, False, False]

![Fork, chain, collider](images/causality_assets/fig4_fork_chain_collider.png)

For Figure 4 we see a fork and chain are open until their middle node is conditioned on; a collider is closed until conditioning opens it.

Here are the three junctions. Shaded nodes are conditioned on. Only the collider opens when we condition on it or a descendant.

So an extra control can remove bias (the fork), change the question we answer (the chain), or create bias (the collider). The variable's position in the graph decides which, whatever its predictive power. Try one more graph.

Take fund performance $Q$, manager skill $K$ and whether a fund is listed in a database $L$, with $K\to Q$, $K\to L$ and $Q\to L$. We want the effect of skill on performance. Should we compare all funds, or only listed ones? Decide, then check with d-separation after removing the arrow we want to measure.

In [ ]:
g_kql = nx.DiGraph([("K", "Q"), ("K", "L"), ("Q", "L")])
non_causal = g_kql.copy()
non_causal.remove_edge("K", "Q")
for z in (set(), {"L"}):
    print(f"condition on {sorted(z) or 'nothing'}: K and Q d-separated without the arrow K -> Q: {nx.is_d_separator(non_causal, {'K'}, {'Q'}, z)}")
print("L is a descendant of K:", "L" in nx.descendants(g_kql, "K"))

We should compare all funds and adjust for nothing. Skill has no parents, so no backdoor path exists. Listing $L$ is a collider on $K\to L\leftarrow Q$ and also an effect of $K$, so conditioning on it opens a non-causal path, as the check shows.

How do we turn the right choice into a number?

## From the graph to a number

The graph says to hold $C$ fixed. We compare funds with the same $C$ and average those comparisons over the population we care about. This is where conditional exchangeability enters: within each level of $C$, the policy a fund chose carries no extra information about the losses it would have had under either policy [5, section 2.2]. Funds that chose high leverage then stand in fairly for what the others would have done had they chosen it. It's also called ignorability.

Let $m(d,c)=E[Y\mid D=d,C=c]$ be the observed loss rate among funds with policy $d$ and health $c$. 


The table gives $m(1,1)=0.20$, $m(0,1)=0.10$, $m(1,0)=0.60$ and $m(0,0)=0.50$.

If every fund ran high leverage, healthy funds would lose at 20% and impaired ones at 60%, and the population is half healthy and half impaired, so we weight the two rates by 0.5 each. In symbols,

$$E[Y\mid do(D=d)]=\sum_{c} m(d,c)\,P(C=c).$$

With one binary $C$ this is standardization, and the table gives 40% for high leverage and 30% for low, a gap of $+10$ pp. This is the comparison the committee needed: the same mix of funds under each policy. Now we ask the simulated SCM to make the comparison using observed rows alone. The formula should land on the surgery's risks, though it never sees $Y(1)$ or $Y(0)$.

In [ ]:
def standardize(data):
    counts = data.groupby(["C", "D"]).size().unstack(fill_value=0).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    if (counts == 0).any().any():
        raise ValueError("an empty (C, D) cell, so m(d, c) is undefined there")
    m = data.groupby(["C", "D"])["Y"].mean().unstack()
    p_c = data["C"].value_counts(normalize=True)
    return m, p_c, {d: float((m[d] * p_c).sum()) for d in (0, 1)}

m, p_c, risk = standardize(funds)
tau = risk[1] - risk[0]
print(m.rename(index={1: "Healthy", 0: "Impaired"}, columns={1: "m(1,c)", 0: "m(0,c)"}).to_string(float_format="{:.2f}".format))
print(f"Target weights: healthy {p_c[1]:.2f}, impaired {p_c[0]:.2f}")
print(f"Standardized risk: high {risk[1]:.2f}, low {risk[0]:.2f}, effect {tau:+.2f}   (naive {naive:+.2f})")
assert abs(risk[1] - 0.40) < 1e-12 and abs(risk[0] - 0.30) < 1e-12 and abs(tau - 0.10) < 1e-12

obs = pd.DataFrame({"C": c, "D": d_seen, "Y": y_seen})
*_, adjusted = standardize(obs)
print(f"Simulated SCM: formula on observed rows high {adjusted[1]:.3f}, low {adjusted[0]:.3f}; surgery high {y1.mean():.3f}, low {y0.mean():.3f}")
assert abs(adjusted[1] - y1.mean()) < 0.01 and abs(adjusted[0] - y0.mean()) < 0.01

The standardized $+10$ pp compares the same 50/50 mix of funds under both policies, and the naive $-14$ pp compares whichever funds chose each. On the simulated rows the formula reproduces the surgery within sampling noise. That checks the arithmetic on a model we wrote ourselves. It says nothing about whether the graph is right.

The formula is only a weighted average of observed rates, and nobody set anything in the rows it uses. Why should it answer the policy question?

## Why does the adjusted answer work?

We need to show that the standardized risks, 0.40 for high leverage and 0.30 for low, equal $P(Y=1\mid do(D=d))$. We will make three moves, one per step. The first is ordinary probability; the other two use do-calculus, rules for rewriting a probability that contains $do(\cdot)$ when d-separation holds in an edited graph [2, 3, 4]. If this is your first encounter with do-calculus, follow the reason for each graph edit; the rule numbers are labels you can return to later. When the rewriting ends with observed distributions only, the causal quantity is identified.

**Step 1. Split by funding health.** Start with the target, the loss probability when every fund is set to policy $d$. The law of total probability says it equals the loss probability within each health group, weighted by that group's share, and this holds in the world where $D$ was set just as it holds in the world we observed.

$$P(Y=1\mid do(D=d))=\sum_c P(Y=1\mid do(D=d),C=c)\,P(C=c\mid do(D=d)).$$

This is arithmetic and uses no graph. Two factors still carry a $do$. The second, $P(C=c\mid do(D=d))$, is the share of healthy funds after we set the policy. The first is the loss rate inside a health group after we set it.

**Step 2. Setting leverage can't change health.** Funding health was fixed before the decision, so a policy set today shouldn't move the share of healthy funds: $P(C=c\mid do(D=d))$ should equal $P(C=c)$. The simulation showed it, with 0.5 healthy under intervention against 0.8 among the funds observed with $D=1$.

Rule 3 is the graph test behind that intuition. It lets us delete a $do$ when the variable acted on has no open path to the quantity of interest, checked in a graph edited to match the intervention. Setting $D$ deletes the arrows into $D$, so we test on that graph, written $G_{\overline D}$ (Figure 3, right). Here the quantity of interest is $C$, and the question is whether $C$ and $D$ are d-separated in $G_{\overline D}$.

The only path left there, $C\to Y\leftarrow D$, is blocked at the collider $Y$. So the test should pass in $G_{\overline D}$ and fail in the original graph, where $C\to D$ is a direct arrow.

In [ ]:
def cut_in(g, nodes):
    h = g.copy()
    h.remove_edges_from([e for e in g.edges if e[1] in nodes])
    return h

def cut_out(g, nodes):
    h = g.copy()
    h.remove_edges_from([e for e in g.edges if e[0] in nodes])
    return h

g = nx.DiGraph(EDGES)
rule3 = pd.DataFrame([(name, nx.is_d_separator(h, {"C"}, {"D"}, set())) for name, h in [("original graph G", g), ("arrows into D deleted", cut_in(g, {"D"}))]], columns=["graph", "C and D d-separated"])
print(rule3.to_string(index=False))
assert rule3["C and D d-separated"].tolist() == [False, True]

![Rule 3 graphical test](images/causality_assets/fig5_rule3_graphs.png)

Rule 3 is checked in figure 5 after deleting arrows into the intervened-on leverage variable.

The figure above shows why we can drop $do$ from the health factor (Rule 3). Left: the original graph, where $C$ and $D$ are directly linked. Right: with the arrow into $D$ deleted (dotted), the only path is $C\to Y\leftarrow D$, blocked at the collider $Y$.

The test passes in the edited graph and fails in the original, so Rule 3 gives $P(C=c\mid do(D=d))=P(C=c)$.

**Step 3. Seeing can replace setting once health is held fixed.** The factor left is $P(Y=1\mid do(D=d),C=c)$, the loss rate in health group $c$ when the policy is set. Our data give $P(Y=1\mid D=d,C=c)$, where the policy was chosen. Inside a health group, funds that chose high leverage should stand in for funds that were set to it, unless a hidden common cause of leverage and losses exists.

Rule 2 makes this precise. Recall that a backdoor path from $D$ to $Y$ starts with an arrow into $D$. Rule 2 allows the swap when what we've observed, here $C$, blocks every one of them. To test that, we delete the arrows that leave $D$. That removes $D\to Y$ and leaves only backdoor paths. Call this graph $G_{\underline D}$. Its only path between $D$ and $Y$ is the fork $D\leftarrow C\rightarrow Y$, which conditioning on $C$ should block.

We also run two failures: conditioning on nothing, and a hidden common cause $H$ of leverage and losses (say, private information), which should reopen the path $C$ closed. The real test should pass and both failures should fail.

In [ ]:
g_hidden = nx.DiGraph(EDGES + [("H", "D"), ("H", "Y")])
tests = [("arrows out of D deleted", cut_out(g, {"D"}), {"C"}), ("arrows out of D deleted", cut_out(g, {"D"}), set()), ("hidden cause H added", cut_out(g_hidden, {"D"}), {"C"})]
rule2 = pd.DataFrame([(name, ", ".join(sorted(z)) or "nothing", nx.is_d_separator(h, {"Y"}, {"D"}, z)) for name, h, z in tests], columns=["graph", "condition on", "Y and D d-separated"])
print(rule2.to_string(index=False))
assert rule2["Y and D d-separated"].tolist() == [True, False, False]

![Rule 2 graphical test](images/causality_assets/fig6_rule2_graphs.png)

Figure 6 show what happens when $C$ is held fixed (Rule 2). Left: arrows out of $D$ deleted (dotted) and $C$ conditioned on (shaded). The fork $D\leftarrow C\rightarrow Y$ is blocked. Right: a hidden common cause $H$ (dashed) opens $D\leftarrow H\rightarrow Y$, so $C$ no longer blocks every path.*

The real test passed and both failures failed, so Rule 2 gives $P(Y=1\mid do(D=d),C=c)=P(Y=1\mid D=d,C=c)$. With the hidden $H$ the test fails, the proof can't take its Rule 2 step (the third line of the block below), and nothing guarantees the formula equals the intervention probability.

Chained together, the three moves give the derivation. With $P(C=1)=P(C=0)=0.5$, the four rates from the table then give the two numeric lines at the bottom of the same block.

$$
\begin{aligned}
P(Y=1\mid do(D=d))
&=\sum_c P(Y=1\mid do(D=d),C=c)\,P(C=c\mid do(D=d)) &&\text{(total probability)}\\
&=\sum_c P(Y=1\mid do(D=d),C=c)\,P(C=c) &&\text{(Rule 3)}\\
&=\sum_c P(Y=1\mid D=d,C=c)\,P(C=c) &&\text{(Rule 2)}\\[6pt]
P(Y=1\mid do(D=1))&=0.5(0.20)+0.5(0.60)=0.40\\
P(Y=1\mid do(D=0))&=0.5(0.10)+0.5(0.50)=0.30
\end{aligned}
$$

The third line uses only distributions we can read from observed data. That is identification: a causal quantity written in terms of observable distributions. The two numeric lines give 40% for high leverage and 30% for low.

The $+10$ pp gap is the intervention effect if five conditions hold.

1. The graph is right, and each variable is a function of its parents plus independent noise.
2. No residual confounding given $C$ (conditional exchangeability).
3. Consistency: a fund's observed outcome equals its outcome under the policy it followed, $Y=Y(D)$.
4. Positivity: both policies occur at every level of $C$.
5. No interference between funds.

Rule 1, which drops an observation d-separated from the outcome, wasn't needed.

The derivation is only as good as these conditions. They can fail in two ways. The data may not cover every cell the formula needs, or the graph itself may be wrong.

## What if a comparison is missing?

The first failure is about coverage. The proof is an equality between distributions and says nothing about whether the data cover every cell. Positivity requires both policies at every level of $C$, $P(D=d\mid C=c)>0$ [5, section 3.3]. To see it fail, we delete the impaired, low-leverage cell. The pooled gap should compute as usual, and `standardize` should refuse.

In [ ]:
gap = funds[~((funds["C"] == 0) & (funds["D"] == 0))]
print(gap.groupby(["C", "D"]).size().unstack(fill_value=0).rename(index={1: "Healthy", 0: "Impaired"}, columns={1: "high leverage", 0: "low leverage"}).to_string())
print(f"Pooled gap still computes: {gap[gap.D == 1].Y.mean() - gap[gap.D == 0].Y.mean():+.2f}")
try:
    standardize(gap)
except ValueError as err:
    print("standardize refused:", err)

The pooled gap returns a number without complaint, while the adjusted estimate has nothing to stand on, since $m(0,\text{impaired})$ doesn't exist. If impaired funds never choose low leverage, more rows won't fill the gap. If the cell is empty by chance, more data can.

The proof is a graph question and positivity is a data question. Identification rewrites a causal quantity in observable terms, which is what the proof did. Estimation computes a number from finite rows, which is what we did with the 1,000 funds. A perfect estimate of $m(d,c)$ settles estimation and leaves identification untouched. That raises the harder worry. Suppose positivity holds, every calculation runs and every test passes. Could the answer still be wrong?

## Could the same data tell a different story?

Say we set two rival accounts of the same funds side by side. 

In the first, leverage raises the chance of a large loss, so cutting leverage would protect funds. In the second, a hidden fund trait, say a manager's unrecorded appetite for risk, drives both leverage and losses. Leverage does nothing, and cutting it would change nothing.

The two stories predict the same kind of data. Funds with high leverage lose money more often, and a fund's leverage tells us a lot about its losses. They part ways only on the committee's question: what happens if we set the policy?

Suppose someone hands you a million rows of $(D,Y)$ from one of the two worlds. Could any observational estimator tell which? How well should $D$ predict $Y$ in each? Think first, then we build both worlds and compute exactly.

To make the two stories exact, shrink them to two binary variables. A hidden fair coin $T$ stands for the trait: a Bernoulli(0.5) variable, equal to 1 with probability 0.5. Noise $B$ is an independent Bernoulli(0.1) variable, so it's 1 about once in ten.

We need one operation, exclusive-or, written $a\oplus b$. It equals 1 when exactly one of $a,b$ is 1, so it tells us whether the two differ. Then $a\oplus B$ is $a$ itself nine times in ten and $a$ flipped the rest of the time.

Model A is the first story: leverage copies the coin and losses follow leverage. Model B is the second story: leverage does nothing and the coin drives both. Side by side:

$$
\begin{aligned}
\text{Model A:}\quad D&=T, & Y&=D\oplus B\\
\text{Model B:}\quad D&=T, & Y&=T\oplus B
\end{aligned}
$$

The graph for B is $D\leftarrow T\rightarrow Y$, a hidden common cause, and $D$ is a deterministic copy of $T$. We enumerate both models exactly with fractions.

**Predict.** Work out $P(Y=1\mid do(D=1))$ and $P(Y=1\mid do(D=0))$ in each model before running the cell.

In [ ]:
def model_a(u, b, do=None):
    d = u if do is None else do
    return d, d ^ b

def model_b(u, b, do=None):
    d = u if do is None else do
    return d, u ^ b

def enumerate_model(model):
    joint, do = {}, {0: F(0), 1: F(0)}
    for u, b in product((0, 1), repeat=2):
        p = F(1, 2) * (F(1, 10) if b else F(9, 10))
        joint[model(u, b)] = joint.get(model(u, b), F(0)) + p
        for d in (0, 1):
            do[d] += p * model(u, b, do=d)[1]
    return joint, do

joint_a, do_a = enumerate_model(model_a)
joint_b, do_b = enumerate_model(model_b)
table = pd.DataFrame({
    "Model A": {f"D={d}, Y={y}": str(joint_a[(d, y)]) for d, y in sorted(joint_a)},
    "Model B": {f"D={d}, Y={y}": str(joint_b[(d, y)]) for d, y in sorted(joint_b)},
})
effects = pd.DataFrame({"Model A": {"P(Y=1 | do(D=1))": str(do_a[1]), "P(Y=1 | do(D=0))": str(do_a[0]), "causal effect": str(do_a[1] - do_a[0])},
                        "Model B": {"P(Y=1 | do(D=1))": str(do_b[1]), "P(Y=1 | do(D=0))": str(do_b[0]), "causal effect": str(do_b[1] - do_b[0])}})
print(pd.concat([table, effects]).to_string())
print("Rows 1-4: observational joint P(D,Y). Rows 5-7: interventional.")
print("Accuracy of predicting Y with D:", sum(p for (d, y), p in joint_a.items() if d == y), sum(p for (d, y), p in joint_b.items() if d == y))
assert joint_a == joint_b
assert do_a[1] == F(9, 10) and do_a[0] == F(1, 10) and do_b[1] == do_b[0] == F(1, 2)

The joint distributions match cell for cell, so every dataset is equally likely under both models, and $D$ predicts $Y$ correctly 90% of the time in each. Predictive accuracy is a property of that joint table. The causal effect is a property of what happens when $D$ is set, and the table never records that. So no estimator, however accurate, can separate the models from the data alone. The causal rows part ways: setting $D=1$ gives a loss probability of $0.9$ in Model A and $0.5$ in Model B, so the effects are $0.8$ and $0$. The models are observationally equivalent and causally different.

**Transfer.** Could a hidden trait, with leverage doing nothing, reproduce all four cells of the 1,000-fund table? What would the adjustment formula report?

**Answer.** Yes, and Appendix A constructs it. The adjustment formula would still report $+10$ pp, because it only sees the observed cells. An assumption such as no hidden common cause given $C$, or an actual intervention, breaks the tie.

It is tempting to hope that a better regression, or a flexible machine-learning model, could settle the disagreement. Why can't it?

## What regression estimates

Leave out a variable that affects both the regressor and the outcome, and its effect leaks into the coefficient you do estimate. That's omitted-variable bias (OVB). Our table satisfies the additive model $E[Y\mid D,C]=0.50+0.10\,D-0.40\,C$ exactly, so ordinary least squares (OLS) on $D$ and $C$ should return $+0.10$ on $D$, the effect $\tau$. Leave $C$ out and the $D$ coefficient absorbs health's effect ($-0.40$) times how strongly health tracks leverage ($0.15/0.25=0.6$), giving $0.10-0.24=-0.14$. Appendix B shows the algebra. OLS should return $-0.14$ without $C$ and $+0.10$ with it.

In [ ]:
naive_fit = sm.OLS(funds["Y"], sm.add_constant(funds[["D"]])).fit()
adjusted_fit = sm.OLS(funds["Y"], sm.add_constant(funds[["D", "C"]])).fit()
print(pd.DataFrame({"Y ~ D": naive_fit.params, "Y ~ D + C": adjusted_fit.params}).to_string(float_format="{:.2f}".format, na_rep="-"))
cov_dc = np.cov(funds["D"], funds["C"], ddof=0)[0, 1]
var_d = np.var(funds["D"], ddof=0)
print(f"Slope of C on D = {cov_dc / var_d:.2f}, rebuilt naive slope = {adjusted_fit.params['D'] + adjusted_fit.params['C'] * cov_dc / var_d:.2f}")
assert abs(naive_fit.params["D"] - (-0.14)) < 1e-12 and abs(adjusted_fit.params["D"] - 0.10) < 1e-12
assert abs(cov_dc - 0.15) < 1e-12 and abs(var_d - 0.25) < 1e-12

OLS reproduces the reversal, and the numbers above rebuild the first coefficient from the second. We report no standard errors because the 1,000 rows are the whole enumerated population we constructed, so there's no sampling error here. A real sample would need them. With many covariates, a flexible learner such as a random forest could estimate $m(d,c)$ where a linear model would be misspecified, and the identifying assumptions would be untouched.

**Question.** Why could the adjusted coefficient still be wrong if managers act on private information?

**Answer.** Exchangeability fails, whatever the model's accuracy. A private signal that moves both leverage and losses leaves funds in the same $C$ group differing in an unrecorded way, the hidden $H$ of the Rule 2 test. A machine-learning model can estimate $m(d,c)$ well and still can't establish ignorability, the no-extra-information condition from earlier. Cross-validated accuracy can't tell Model A from Model B or a confounder from a collider.

Finance meets the same ambiguity whenever a price reacts to several kinds of news at once, and researchers deal with it by adding assumptions on purpose. How do they choose them?

### A real financial-econometrics connection

The fund data were constructed to make the logic visible. A related identification problem appears in central-bank announcements. A surprise change in interest rates can reflect both a monetary-policy action and information the bank conveys about its economic outlook. Market prices react to both, so observing rates and equity prices move together does not by itself isolate a pure policy shock. Jarociński and Karadi (2020) [7] and Nakamura and Steinsson (2018) [9] study versions of this problem using additional economic and statistical assumptions. Their designs are more sophisticated than our fund DAG, but the lesson is the same: **identifying a cause requires a defensible source of variation, not just a strong correlation.**

## Conclusion

Back to the risk committee. Its table said high leverage came with 14 fewer points of large-loss risk, and the vote looked easy. Three separate questions sat behind that vote.

Observation asks which funds chose high leverage and how they fared; the pooled gap answers that descriptive question. Identification asks whether the data, together with a graph we're willing to defend, can tell us what would happen if the committee set the policy. Under our graph they can: 40% against 30%, so high leverage adds about 10 points of large-loss risk. Estimation asks how well we computed that number from finite rows, which is where positivity, sample size and the choice of OLS or a forest matter. Better estimation can't repair a failure of identification, and that's why Models A and B stay apart however well we predict.

So the committee shouldn't vote on the pooled figure. It should first ask how funds came to choose their leverage, and whether anything unrecorded drove both the choice and the losses. If it can defend those assumptions, the adjusted comparison speaks to its decision. If it can't, it needs an experiment or a stronger design. The fund population is synthetic. It shows how the argument works and says nothing about what leverage does to real funds.

## References

1. Pearl, Glymour, and Jewell (2016). *Causal Inference in Statistics: A Primer*. https://bayes.cs.ucla.edu/PRIMER/
2. Pearl (1995). *Causal Diagrams for Empirical Research*. Biometrika 82(4), 669-688. https://doi.org/10.1093/biomet/82.4.669
3. Pearl (1995). *A Causal Calculus for Statistical Research*. Pre-proceedings of the Fifth International Workshop on Artificial Intelligence and Statistics, PMLR R0, 430-449. https://proceedings.mlr.press/r0/pearl95a.html
4. Pearl (2009). *Causality: Models, Reasoning, and Inference*, 2nd ed., section 3.4.2. https://bayes.cs.ucla.edu/BOOK-2K/
5. Hernan and Robins. *Causal Inference: What If*. https://miguelhernan.org/whatifbook
6. Rubin (1974). *Estimating Causal Effects of Treatments in Randomized and Nonrandomized Studies*. https://doi.org/10.1037/h0037350

# GitHub

This article is also available on [GitHub](https://github.com/adamd1985/quant_research/blob/main/causality_finance/trading_correlation_with_causation.ipynb)

Kaggle notebook available [here](https://www.kaggle.com/code/addarm/trading-correlation-with-causation)

# Media

All media used (in the form of code or images) are either solely owned by me, acquired through licensing, or part of the Public Domain and granted use through Creative Commons License.

# CC Licensing and Use

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.

**Reproducibility note.** Tested as a Python notebook. Plotting assistance was provided by Claude Sonnet and the resulting calculations and graphical claims should be independently checked.
